In [1]:
# 1. Install YOLO26 and UI dependencies
!pip install ultralytics moviepy gradio -q
!apt-get install ffmpeg -y -q

import cv2
import gradio as gr
from ultralytics import YOLO
from moviepy.editor import VideoFileClip
import numpy as np
import base64
import io
from PIL import Image

# Load the 2026 Small model (NMS-Free & Optimized for speed)
model = YOLO('yolo26s.pt')
print("YOLO26 Small Loaded. Ready for end-to-end detection!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 37.7 MB/s eta 0:00:00
Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 51 not upgraded.
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:294: SyntaxWarning: invalid escape sequence '\d'
  lines_video = [l for l in lines if ' Video: ' in l and re.search('\d+x\d+', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:367: SyntaxWarning: invalid escape sequence '\d'
  rotation_lines = [l for l in lines if 'rotate          :' in l and re.search('\d+$', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:370: SyntaxWarning: invalid escape sequence '\d'
  match = re.search('\d+$', rotation_line)
  if event.key is 'enter':



YOLO26 Small Loaded. Ready for end-to-end detection!


In [2]:
def predict_image(img, conf):
    results = model.predict(img, conf=conf)
    return results[0].plot()

def predict_video(video_path, conf_threshold):
    clip = VideoFileClip(video_path)

    def process_frame(frame):
        # YOLO26 expects BGR, MoviePy gives RGB
        frame_bgr = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)
        results = model.predict(frame_bgr, conf=conf_threshold, verbose=False)
        return cv2.cvtColor(results[0].plot(), cv2.COLOR_BGR2RGB)

    output_filename = "yolo26_detected_video.mp4"
    out_clip = clip.fl_image(process_frame)
    out_clip.write_videofile(output_filename, codec="libx264", audio=False)
    return output_filename

In [3]:
with gr.Blocks(theme=gr.themes.Soft()) as demo:

    gr.Markdown("# 🚀 YOLO26 Engineering Dashboard")

    conf_slider = gr.Slider(0.1, 1.0, value=0.25, label="Confidence Threshold")



    with gr.Tabs():

        with gr.TabItem("🖼️ Image Analysis"):

            with gr.Row():

                img_in = gr.Image(type="numpy")

                img_out = gr.Image()

            btn_i = gr.Button("Detect")

            btn_i.click(predict_image, [img_in, conf_slider], img_out)



        with gr.TabItem("🎥 Video Processing"):

            with gr.Row():

                vid_in = gr.Video()

                vid_out = gr.Video()

            btn_v = gr.Button("Process & Render")

            btn_v.click(predict_video, [vid_in, conf_slider], vid_out)



    gr.Markdown("> **Note:** For Real-Time Webcam, stop this cell and run the 'Webcam Mode' cell below.")



demo.launch(share=True)

  with gr.Blocks(theme=gr.themes.Soft()) as demo:



Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5c6f8bdab3d8f7d74f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [5]:
from google.colab.output import eval_js
from IPython.display import Javascript, display

# JavaScript Bridge for Zero-Latency Webcam
def video_stream():
  js = Javascript('''
    var video; var div = null; var stream; var captureCanvas; var imgOverlay;
    var pendingResolve = null; var shutdown = false;

    async function createDom() {
      if (div !== null) return;
      div = document.createElement('div'); div.style.border = '2px solid black';
      div.style.padding = '10px'; div.style.backgroundColor = 'white'; div.style.width = 'fit-content';
      const title = document.createElement('h3'); title.innerText = "YOLO26 Real-Time";
      div.appendChild(title);
      video = document.createElement('video'); video.style.display = 'none';
      video.width = 640; video.height = 480; video.autoplay = true; div.appendChild(video);
      imgOverlay = document.createElement('img'); imgOverlay.style.display = 'block';
      imgOverlay.width = 640; imgOverlay.height = 480; div.appendChild(imgOverlay);
      captureCanvas = document.createElement('canvas'); captureCanvas.width = 640; captureCanvas.height = 480;
      const stopBtn = document.createElement('button'); stopBtn.innerText = "Stop Stream";
      stopBtn.onclick = () => { shutdown = true; }; div.appendChild(stopBtn);
      document.body.appendChild(div);
      stream = await navigator.mediaDevices.getUserMedia({video: true});
      video.srcObject = stream; await video.play();
      window.requestAnimationFrame(onAnimationFrame);
    }

    function onAnimationFrame() {
      if (!shutdown) window.requestAnimationFrame(onAnimationFrame);
      if (pendingResolve) {
        captureCanvas.getContext('2d').drawImage(video, 0, 0, 640, 480);
        let result = captureCanvas.toDataURL('image/jpeg', 0.8);
        let lp = pendingResolve; pendingResolve = null; lp(result);
      }
    }

    async function stream_frame(imgData) {
      if (shutdown) { stream.getTracks().forEach(t => t.stop()); div.remove(); shutdown = false; return null; }
      if (div === null) await createDom();
      if (imgData) imgOverlay.src = imgData;
      return new Promise(resolve => { pendingResolve = resolve; });
    }
    ''')
  display(js)

# Execution Loop
video_stream() # Call video_stream to initialize the JS functions in the browser
last_img = ""
try:
  while True:
    js_reply = eval_js(f'stream_frame("{last_img}")')
    if not js_reply: break

    # 1. Convert to Image
    image_bytes = base64.b64decode(js_reply.split(',')[1])
    frame = cv2.cvtColor(np.array(Image.open(io.BytesIO(image_bytes))), cv2.COLOR_RGB2BGR)

    # 2. YOLO26 Inference
    res = model.predict(frame, conf=0.25, verbose=False)

    # 3. Overlay & Return
    _, buffer = cv2.imencode('.jpg', res[0].plot())
    last_img = f"data:image/jpeg;base64,{base64.b64encode(buffer).decode('utf-8')}"
except Exception as e:
  print(f"Stopped: {e}")

<IPython.core.display.Javascript object>